In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import sys
import json
import random 
random.seed(42)
import numpy as np 
import pandas as pd 
import librosa as lb


import soundfile as sf
import kagglehub 
import matplotlib.pyplot as plt
from IPython.display import Audio
from tqdm import tqdm

import torch
import torchaudio
import torchaudio.transforms as T
# Kaggle Set-up
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
#os.environ['HF_TOKEN'] = user_secrets.get_secret("hf_access")
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("kgg_user")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("kgg_key")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# TRAIN DATASET
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
STEMS = {'vocals.wav','other.wav','bass.wav','drums.wav'} 
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
SONG_INDEX = ''  

# NOISE DATASET
root_dir = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/'
csv_file = '/meta/esc50.csv'
audio_folder = '/audio/'


SR = 22050
DURATION = 30

#============ Check for GPU availability ==============================#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random = random.Random(RANDOM_SEED)
if torch.cuda.is_available():
    print("GPU is available.Setting RANDOM_SEED .... ")
        # setting for both CPU and GPU
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")
print("\n✅ Environment setup complete!")

import warnings
warnings.filterwarnings("ignore")


🚀 Using device: cpu
   Running on CPU

✅ Environment setup complete!


# Defined Utility

In [30]:
def trim_or_pad(y,sr=SR,duration=DURATION):
    LENGTH = SR*DURATION
    # Trim or pad
    if y.shape[0] >= LENGTH:
        return y[:LENGTH]
    else:
        padding = LENGTH - y.shape[0]
        return np.pad(y,(0,padding))
        
def load_and_fix(path,sr=SR,duration=DURATION):
    LENGTH = sr*duration    
    y,sr = lb.load(path,sr=SR)
    
    # Trim or pad
    if y.shape[0] >= LENGTH:
        return y[:LENGTH]
    else:
        padding = LENGTH - y.shape[0]
        return np.pad(y,(0,padding))

def build_dataset(root_dir,test_split=0.20):
    """
        This takes root dirstory and load paths of all stem files as a dictionary.
        Returns a Dictionary.
    """
    global SR
    global DURATION
    LENGTH = SR*DURATION
    
    # Initialize empty dictionaries
    train_song = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_song = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    
    total_stem_file = 0
    val = 0
    train = 0
    for g in tqdm(GENRES,desc="Loading stems into dict.."):
        folder_path = os.path.join(root_dir,g)
        if os.path.exists(folder_path) and os.path.isdir(folder_path):
            for i in range(0,100):
                song_folder = g + '.' + f"{i:05d}"
                for s in STEMS:
                    stem_file_path = os.path.join(folder_path ,song_folder ,s)
                    if os.path.exists(stem_file_path):
                        total_stem_file += 1
                        y = load_and_fix(stem_file_path)
                        if i >= 100 - int(100*test_split):
                            val += 1
                            val_song[g][s.replace('.wav', '')].append(y)
                        else:
                            train += 1
                            train_song[g][s.replace('.wav', '')].append(y)
                    else:
                        print(f"Stem file {s} not exists !")
            
        else:
            print(f"Folder '{g}' not exists !")

    print("✅ Total number of stems music file loaded successfully : ", total_stem_file)
    print("Validation : ",val,' | ',"Train : ",train)
    return train_song,val_song

def load_noise_audios(root_dir,audio_folder,csv_file):
    """
     It will extract all noise audio waveform and store in a dictionary.
    """
    noise = {}
    df_noise = pd.read_csv(root_dir+csv_file)
    total_noise = 0
    
    for i in tqdm(df_noise.index,desc="Loading Noise audio ... "):
        filename = df_noise.loc[i]['filename']
        path = root_dir+audio_folder+filename
        y = load_and_fix(path,sr=SR,duration=5)    
        noise[filename] = y
        total_noise += 1
    print(f"Total noise files loaded : {total_noise}")
    return noise



# ====================================================================================================== #
def random_time_shift(stem, sr, max_shift_ms=40):

    max_shift = int(sr * max_shift_ms / 1000)
    shift = random.randint(-max_shift, max_shift)

    if shift > 0:
        stem = np.pad(stem, (shift, 0))[:-shift]
    elif shift < 0:
        stem = np.pad(stem, (0, -shift))[-shift:]
    return stem
    
def aug_stem(stem,sr=SR):
    #1. random gain
    random_gain = random.uniform(0.95,1.2)
    stem = stem*random_gain
    
    #2. pitch shift
    if random.random() < 0.5:
        n_steps = random.uniform(-2, 2)
        stem = lb.effects.pitch_shift(stem,sr=sr, n_steps=n_steps)
    #3. time stratch
    if random.random() < 0.5:
        rate = random.uniform(0.8 , 1.4) #0.8 , 1.3
        stem = trim_or_pad(lb.effects.time_stretch(stem,rate=rate))

    #4. time shift
    stem = random_time_shift(stem,sr)
    #5. small noise
    noise = np.random.randn(*stem.shape) * 0.0002
    stem = stem + noise
    stem = stem / np.abs(stem).max()
    
    return stem


def create_single_track(stems):
    """
        Combine 4 stems into a single normalized track.
        All tracks are resampled, padded or trimmed to 30 seconds.
        GPU-compatible.
    """
    weights = np.array([
        np.random.uniform(1,1),
        np.random.uniform(1,1),
        np.random.uniform(1,1),
        np.random.uniform(1,1)
    ])
    weights = weights / weights.sum()
    
    #mix = weights[0]*stems['vocals'] + weights[1]*stems["drums"] + weights[2]*stems["bass"] + weights[3]*stems["other"]
    mix = stems['vocals'] + stems["drums"] + stems["bass"] + stems["other"]
    
    max_amplitude_val = np.abs(mix).max()
    if(max_amplitude_val > 1):
        track = mix/max_amplitude_val
    else:
        return mix
    return track


def stem_recombination(sl,genre):  
    """
        This function randomly pick different stems from same genre and makes a mashup.
        RETURN : single music track (Combination of stems files of different songs from same genre)
    """
    stems = {}
    for s in STEM_KEYS:
        stem = random.choice(sl[genre][s])
        #stems[s] = aug_stem(stem)
        stems[s] = stem
    mashup = create_single_track(stems)
    return mashup
    
def add_noise(mashup,noises,snr_db = 20):
    """
        It will take recombination music and add noise at random places.
    """
    num_insertions = random.choice([4,5]) 
    noise_audios = random.choices(noises,k=num_insertions)
    
    max_start = 22050*(30-5)  # as sample rate for all is 22050 and noise duration is 5sec and audio duration is 30sec.
    
    positions = random.sample(range(0, max_start), num_insertions)
    #print(num_insertions, noise_files, max_start,np.array(positions)/22050)
    
    song_rms = np.sqrt(np.mean(mashup**2))
    output = mashup.copy()
    for i,pos in enumerate(positions):
        #redusing noise volume a little
        noise = noise_audios[i]
        noise_rms = np.sqrt(np.mean(noise**2))
        target_noise_rms = song_rms / (10**(snr_db / 20))
        noise = noise * (target_noise_rms / noise_rms)
        
        output[pos:pos+len(noise)] += noise
    output = output / np.abs(output).max()
    return output

# Data augmentation function
def data_augmentation(train_stems,noises=None,genre=None,sample_count=1,_type='train'):
    """
        Creating actual noisy mashup.
    """
    if genre is None:
        print("ERROR : Provide genre to augment.")
        return 
    total_size = 0 #Mb

    print(genre)
    # Create directory if it doesn't exist
    try:
        # Directory path
        if _type=='train':
            dir_path = f"/kaggle/working/{genre}-train"
        else:
            dir_path = f"/kaggle/working/{genre}-val"    
        os.makedirs(dir_path, exist_ok=True)
        print(f"Directory created at: {dir_path}")
    except Exception as e:
        print(f"Error creating directory: {e}")
    
    for i in tqdm(range(0,sample_count),desc="Creating noisy mashup"):
        mashup = stem_recombination(train_stems,genre)
        #noisy_mashup = add_noise(mashup,noises,snr_db=8)
        sf.write(f"{dir_path}/mashup_v2_{i}.wav", mashup, SR)
        total_size += os.path.getsize(f"{dir_path}/mashup_v2_{i}.wav")
    print(f"🤺 {total_size/(1024**3)} Gb of memory is used for storing augmented music for `{genre}` genre.")

print("✅ Defined successfully.")

✅ Defined successfully.


# Music augmentation

## - Stem files loading  

In [3]:
train_stems,val_stems = build_dataset(DATA_ROOT)
#noise = load_noise_audios(root_dir,audio_folder,csv_file)

Loading stems into dict..: 100%|██████████| 10/10 [08:49<00:00, 52.93s/it]

✅ Total number of stems music file loaded successfully :  4000
Validation :  800  |  Train :  3200


In [ ]:
#noise_values = list(noise.values())
#train_noise,val_noise =noise_values[:1000],noise_values[1000:]
#print(len(train_noise),len(val_noise))

In [28]:
# TESTING 
mashup = stem_recombination(train_stems,'reggae')
#mashup = add_noise(mashup,list(noise.values()),snr_db=8)
Audio(mashup,rate=SR)

In [9]:
mashup.shape

(661500,)

## - Train data creation

In [86]:
idx = 9
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
g = GENRES[idx]
data_augmentation(train_stems,genre=g,sample_count=5000)

rock
Directory created at: /kaggle/working/rock-train


Creating noisy mashup: 100%|██████████| 5000/5000 [02:16<00:00, 36.53it/s]

🤺 6.160903722047806 Gb of memory is used for storing augmented music for `rock` genre.


In [87]:
# uploading
handle = f'akashkumbhakar/{g}-augmented-5000-mashup-train'
train_local_dataset= f'/kaggle/working/{g}-train'

# Create a new dataset
kagglehub.dataset_upload(
    handle, 
    train_local_dataset,
    version_notes="Added some noise free train sample."
)


Uploading Dataset https://api.kaggle.com/datasets/akashkumbhakar/rock-augmented-5000-mashup-train ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpslymtcrh/archive.zip


Uploading: 100%|██████████| 6.62G/6.62G [02:20<00:00, 47.2MB/s]  

Upload successful: /tmp/tmpslymtcrh/archive.zip (6GB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/akashkumbhakar/rock-augmented-5000-mashup-train


In [88]:
import shutil
shutil.rmtree(train_local_dataset)
print(f"{train_local_dataset}  removed")

/kaggle/working/rock-train  removed


## - Validation set creation

In [89]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
g = GENRES[idx]
data_augmentation(val_stems,genre=g,sample_count=500,_type='val')

rock
Directory created at: /kaggle/working/rock-val


Creating noisy mashup: 100%|██████████| 500/500 [00:10<00:00, 45.87it/s]

🤺 0.6160903722047806 Gb of memory is used for storing augmented music for `rock` genre.


In [90]:
# Storing to kaggle hub
handle = f'akashkumbhakar/{g}-augmented-500-mashup-val'
val_local_dataset= f'/kaggle/working/{g}-val'

# Create a new dataset
kagglehub.dataset_upload(
    handle,
    val_local_dataset,
    version_notes="Added some noise free val sample."
    
)

Uploading Dataset https://api.kaggle.com/datasets/akashkumbhakar/rock-augmented-500-mashup-val ...
More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmpvab4tb93/archive.zip


Uploading: 100%|██████████| 662M/662M [00:15<00:00, 42.8MB/s] 

Upload successful: /tmp/tmpvab4tb93/archive.zip (631MB)


Your dataset version has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/akashkumbhakar/rock-augmented-500-mashup-val


In [91]:
import shutil
shutil.rmtree(val_local_dataset)
print(f"{val_local_dataset}  removed")

/kaggle/working/rock-val  removed
